## Side task prediction analysis

In [1]:
import torch
import sys

data = torch.load("../../../../../saved/with_side_task/titles_images_categories_with_side_task.pth", map_location='cpu', weights_only=False)

In [2]:
from recbole.utils.utils import init_seed
from recbole.utils.logger import init_logger
from logging import getLogger

config = data['config']
config.final_config_dict['data_path'] = r'\\wsl.localhost\Ubuntu\home\piotrs\Uni\SequentialRecommendation\recbole\data\dataset\Amazon_Sports_and_Outdoors'
config['device'] = 'cpu'

for attr, value in config.__dict__.items():
    print(f"{attr} = {value}")

init_seed(config['seed'], config['reproducibility'])
init_logger(config)
logger = getLogger()
logger.info(config)

15 Oct 23:55    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 212
state = INFO
reproducibility = True
data_path = \\wsl.localhost\Ubuntu\home\piotrs\Uni\SequentialRecommendation\recbole\data\dataset\Amazon_Sports_and_Outdoors
show_progress = True
save_dataset = False
save_dataloaders = False
benchmark_filename = None

Training Hyper Parameters:
checkpoint_dir = saved
epochs = 200
train_batch_size = 1024
learner = adam
learning_rate = 0.0001
eval_step = 2
stopping_step = 10
clip_grad_norm = None
weight_decay = 0.0
loss_decimal_place = 4

Evaluation Hyper Parameters:
eval_args = {'split': {'LS': 'valid_and_test'}, 'group_by': 'user', 'order': 'TO', 'mode': 'full'}
metrics = ['Recall', 'NDCG']
topk = [3, 5, 10, 20]
valid_metric = Recall@10
valid_metric_bigger = True
eval_batch_size = 256
metric_decimal_place = 4

Dataset Hyper Parameters:
field_separator = 	
seq_separator =  
USER_ID_FIELD = user_id
ITEM_ID_FIELD = item_id
RATING_FIELD = rating
TIME_FIELD = timestamp
s

parameters = {'General': ['gpu_id', 'use_gpu', 'seed', 'reproducibility', 'state', 'data_path', 'benchmark_filename', 'show_progress', 'config_file', 'save_dataset', 'save_dataloaders'], 'Training': ['epochs', 'train_batch_size', 'learner', 'learning_rate', 'training_neg_sample_num', 'training_neg_sample_distribution', 'eval_step', 'stopping_step', 'checkpoint_dir', 'clip_grad_norm', 'loss_decimal_place', 'weight_decay'], 'Evaluation': ['eval_args', 'metrics', 'topk', 'valid_metric', 'valid_metric_bigger', 'eval_batch_size', 'metric_decimal_place'], 'Dataset': ['field_separator', 'seq_separator', 'USER_ID_FIELD', 'ITEM_ID_FIELD', 'RATING_FIELD', 'TIME_FIELD', 'seq_len', 'LABEL_FIELD', 'threshold', 'NEG_PREFIX', 'ITEM_LIST_LENGTH_FIELD', 'LIST_SUFFIX', 'MAX_ITEM_LIST_LENGTH', 'POSITION_FIELD', 'HEAD_ENTITY_ID_FIELD', 'TAIL_ENTITY_ID_FIELD', 'RELATION_ID_FIELD', 'ENTITY_ID_FIELD', 'load_col', 'unload_col', 'unused_col', 'additional_feat_suffix', 'filter_inter_by_user_or_item', 'rm_dup_in

In [3]:
from recbole.data.utils import create_dataset

dataset = create_dataset(config)
logger.info(dataset)

\\wsl.localhost\Ubuntu\home\piotrs\Uni\SequentialRecommendation\recbole\data\dataset\dataset.py:446: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[field].fillna(value='', inplace=True)
\\wsl.localhost\Ubuntu\home\piotrs\Uni\SequentialRecommendation\recbole\data\dataset\dataset.py:446: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on wh

In [4]:
from recbole.data.utils import data_preparation

train_data, valid_data, test_data = data_preparation(config, dataset)

15 Oct 23:55    INFO  [Training]: train_batch_size = [1024] negative sampling: [None]
15 Oct 23:55    INFO  [Evaluation]: eval_batch_size = [256] eval_args: [{'split': {'LS': 'valid_and_test'}, 'group_by': 'user', 'order': 'TO', 'mode': 'full'}]


In [5]:
from recbole.utils.utils import get_model

model = get_model(config['model'])(config, train_data.dataset).to(config['device'])
logger.info(model)

15 Oct 23:55    INFO  SASRecD(
  (item_embedding): Embedding(18358, 256, padding_idx=0)
  (position_embedding): Embedding(50, 256)
  (feature_embed_layer_list): ModuleList(
    (0-1): 2 x Embedding(18358, 128)
    (2): FeatureSeqEmbLayer()
  )
  (trm_encoder): DIFTransformerEncoder(
    (layer): ModuleList(
      (0-2): 3 x DIFTransformerLayer(
        (multi_head_attention): DIFMultiHeadAttention(
          (query): Linear(in_features=256, out_features=256, bias=True)
          (key): Linear(in_features=256, out_features=256, bias=True)
          (value): Linear(in_features=256, out_features=256, bias=True)
          (query_p): Linear(in_features=256, out_features=256, bias=True)
          (key_p): Linear(in_features=256, out_features=256, bias=True)
          (query_layers): ModuleList(
            (0-1): 2 x Linear(in_features=128, out_features=128, bias=True)
            (2): Linear(in_features=64, out_features=64, bias=True)
          )
          (key_layers): ModuleList(
        

In [6]:
model.load_state_dict(data['state_dict'])
model.eval()

SASRecD(
  (item_embedding): Embedding(18358, 256, padding_idx=0)
  (position_embedding): Embedding(50, 256)
  (feature_embed_layer_list): ModuleList(
    (0-1): 2 x Embedding(18358, 128)
    (2): FeatureSeqEmbLayer()
  )
  (trm_encoder): DIFTransformerEncoder(
    (layer): ModuleList(
      (0-2): 3 x DIFTransformerLayer(
        (multi_head_attention): DIFMultiHeadAttention(
          (query): Linear(in_features=256, out_features=256, bias=True)
          (key): Linear(in_features=256, out_features=256, bias=True)
          (value): Linear(in_features=256, out_features=256, bias=True)
          (query_p): Linear(in_features=256, out_features=256, bias=True)
          (key_p): Linear(in_features=256, out_features=256, bias=True)
          (query_layers): ModuleList(
            (0-1): 2 x Linear(in_features=128, out_features=128, bias=True)
            (2): Linear(in_features=64, out_features=64, bias=True)
          )
          (key_layers): ModuleList(
            (0-1): 2 x Linear(

In [7]:
import torch.nn as nn

def predict_side_task(interaction):
        item_seq = interaction[model.ITEM_SEQ]
        item_seq_len = interaction[model.ITEM_SEQ_LEN]
        pos_items = interaction[model.POS_ITEM_ID]

        seq_output = model.forward(item_seq, item_seq_len)

        test_items_emb = model.item_embedding.weight
        scores = torch.matmul(seq_output, test_items_emb.transpose(0, 1))

        idx = torch.argmax(scores, dim=1)

        loss = dict()

        for i, a_predictor in enumerate(model.ap):
            if model.attribute_predictor[i] == 'cos_sim':
                true_emb = model.feature_embed_layer_list[i](pos_items)
                pred_emb = a_predictor(seq_output)

                # Normalize embeddings to unit vectors
                pred_emb_norm = torch.nn.functional.normalize(pred_emb, p=2, dim=-1)
                true_emb_norm = torch.nn.functional.normalize(true_emb, p=2, dim=-1)

                # Compute cosine similarity
                cos_sim = (pred_emb_norm * true_emb_norm).sum(dim=-1)
                attribute_loss = (1 - cos_sim).mean()

                loss[model.selected_features[i]] = attribute_loss
            elif model.attribute_predictor[i] == 'linear':
                attribute_logits = a_predictor(seq_output)
                attribute_labels = interaction.interaction[model.selected_features[i]]
                attribute_labels = nn.functional.one_hot(attribute_labels, num_classes=model.n_attributes[
                    model.selected_features[i]])

                if len(attribute_labels.shape) > 2:
                    attribute_labels = attribute_labels.sum(dim=1)
                attribute_labels = attribute_labels.float()
                attribute_loss = model.attribute_loss_fct(attribute_logits, attribute_labels)
                attribute_loss = torch.mean(attribute_loss[:, 1:])
                loss[model.selected_features[i]] = attribute_loss
            elif model.attribute_predictor[i] == 'not' or model.attribute_predictor[i] == '':
                pass

        return loss

In [8]:
feat_loss = dict({feature: 0 for feature in model.selected_features})

batches = 0

for interaction, _, _, _ in test_data:
    batch_loss = predict_side_task(interaction)

    batches += 1

    for feature in model.selected_features:
        feat_loss[feature] += batch_loss[feature].item()

print(feat_loss)
print(batches)

{'ent_emb': 24.200865402817726, 'img_emb': 53.09848001599312, 'categories': -35.41977010667324}
140
